# 02 | Drought runs and memory associations
Drought is SGI ≤ −1. Missing observations break runs. Complete episodes have observed non-drought boundaries.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import norm, spearmanr
BASE = Path.cwd() if (Path.cwd()/'data_input').exists() else Path.cwd().parent
assert (BASE/'data_input/monthly_heads_m_asl.csv').exists(), 'Open from repo root or notebooks/'
sgi=pd.read_csv(BASE/'data_processed/monthly_sgi.csv');ref=pd.read_csv(BASE/'data_processed/drought_events.csv');memory=pd.read_csv(BASE/'data_processed/groundwater_memory.csv')


In [ ]:
events=[]
for scale in [3,6,12]:
  for well,g in sgi.groupby('well'):
    g=g.sort_values('month').reset_index(drop=True);x=g[f'SGI_{scale}'].to_numpy()
    dry=np.isfinite(x)&(x<=-1)
    starts=np.flatnonzero(dry&~np.r_[False,dry[:-1]])
    ends=np.flatnonzero(dry&~np.r_[dry[1:],False])
    for i,j in zip(starts,ends):
      left=i==0 or not np.isfinite(x[i-1]);right=j==len(x)-1 or not np.isfinite(x[j+1])
      events.append(dict(SGI_scale=scale,Well=well,Start=g.month[i],End=g.month[j],
                         Observed_duration_months=j-i+1,Fully_observed_episode=int(not left and not right),
                         Severity_sum_abs_SGI=-x[i:j+1].sum()))
runs=pd.DataFrame(events)
assert len(runs)==len(ref)
for col in ['SGI_scale','Well','Start','End','Observed_duration_months','Fully_observed_episode']:
    a=runs.sort_values(['SGI_scale','Well','Start'])[col].reset_index(drop=True)
    b=ref.sort_values(['SGI_scale','Well','Start'])[col].reset_index(drop=True)
    pd.testing.assert_series_equal(a,b,check_names=False)
print('Drought runs:',runs.groupby('SGI_scale').size().to_dict())
print('Complete:',runs.groupby('SGI_scale').Fully_observed_episode.sum().to_dict())


In [ ]:
complete=runs.query('Fully_observed_episode==1')
metrics=complete.groupby(['Well','SGI_scale']).agg(events=('Start','count'),mean_duration=('Observed_duration_months','mean'),mean_severity=('Severity_sum_abs_SGI','mean')).reset_index()
crs=[]
for scale in [3,6,12]:
 for well,g in sgi.groupby('well'):
  x=g.sort_values('month')[f'SGI_{scale}'].to_numpy()
  pairs=np.isfinite(x[:-1])&np.isfinite(x[1:])&(x[:-1]<=-1)
  crs.append((well,scale,float(np.sum(pairs&(x[1:]>-1))/np.sum(pairs)) if np.sum(pairs) else np.nan))
metrics=metrics.merge(pd.DataFrame(crs,columns=['Well','SGI_scale','CRS']),on=['Well','SGI_scale'])
q=metrics.merge(memory,left_on='Well',right_on='well').query('group in ["D","M"]')
for scale in [3,6,12]:
 p=q.query('SGI_scale==@scale');print(scale,{k:round(float(spearmanr(p.detrended_M12,p[k]).statistic),4) for k in ['events','mean_duration','mean_severity','CRS']})
